# تحدي مُرقّم — Full pipeline from scratch (3-hour variant)
**Reproduces my 27th-place recipe end-to-end from nothing but the competition data, in a single GPU session.**

This is the compact 4-component version of the full system: AraBERT → pseudo-labels → CAMeLBERT → pseudo-labels → augmented AraBERT + a multi-label head, all combined by a per-class logistic stacker. The full 16-component chain (~20–26 GPU-hours, resumable) is `scripts/10_v32_unified_from_scratch.py` in the repository — this notebook demonstrates the *method* in one 2–3 h T4 session.

**Inputs:** competition `train.csv` / `test.csv` only. **Requires a GPU** (it exits with a message otherwise). Every fold is cached the moment it finishes, so an interrupted session resumes instead of retraining.

Full writeup & all scripts: https://github.com/naz50/muraqqam-challenge-solution

## 1 · Setup

Fixed seeds, automatic Kaggle / Colab / local detection, per-fold cache directory, and a `log()` helper that mirrors everything to `results.txt` (and to Google Drive on Colab).

In [ ]:
import os, random, gc, shutil, inspect
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from collections import Counter

BASE_SEED = 42
random.seed(BASE_SEED); np.random.seed(BASE_SEED); torch.manual_seed(BASE_SEED)

# ===== configuration =====
ON_KAGGLE = os.path.exists('/kaggle/working')   # check Kaggle first: some Kaggle images also contain /content
ON_COLAB  = (not ON_KAGGLE) and os.path.exists('/content')
WORK_DIR = '/kaggle/working' if ON_KAGGLE else ('/content/working' if ON_COLAB else './working')
os.makedirs(WORK_DIR, exist_ok=True)
CKPT_DIR = os.path.join(WORK_DIR, 'ckpt_tmp')

FAST_RUN = False          # True = quick smoke test (not a submission run)
NUM_EPOCHS   = 1 if FAST_RUN else 6
NUM_EPOCHS_L = 1 if FAST_RUN else 6   # the large model saturates early
NF           = 2 if FAST_RUN else 4   # CV folds per family
CONF = 0.90
TRAIN_AG1 = True   # AraBERT + augmentation component
TRAIN_ML  = False
NUM_EPOCHS_ML = 1 if FAST_RUN else 6
AUG_SYNTH_FACTOR = 2    # synthetic docs = 2x the training docs
AUG_RARE_DUP     = 3    # rare-mark windows repeated 3x

M_AR  = 'aubmindlab/bert-base-arabertv02'
M_CA  = 'CAMeL-Lab/bert-base-arabic-camelbert-msa'
M_L   = 'aubmindlab/bert-large-arabertv02'
M_EL  = 'aubmindlab/araelectra-base-discriminator'
M_AB  = ['UBC-NLP/ARBERTv2', 'UBC-NLP/ARBERT']
M_AS  = 'asafaya/bert-base-arabic'

def resolve_model(cands):
    if isinstance(cands, str): return cands
    from transformers import AutoTokenizer as _AT
    for c in cands:
        try:
            _AT.from_pretrained(c); return c
        except Exception:
            continue
    raise ValueError('no model: ' + str(cands))

# ===== Colab: mount Google Drive for persistence across sessions =====
DRIVE_DIR = None
if ON_COLAB:
    try:
        from google.colab import drive as _gdrive
        _gdrive.mount('/content/drive')
        DRIVE_DIR = '/content/drive/MyDrive/muraqqam'
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print('Drive mounted:', DRIVE_DIR)
    except Exception as _e:
        print('Warning: Drive not mounted — saving locally only:', _e)
# on Kaggle the fold cache lives in /kaggle/working and persists with version outputs
CACHE_ROOT = DRIVE_DIR if DRIVE_DIR else WORK_DIR

def _sync_to_drive():
    if not DRIVE_DIR: return
    try:
        for fn in ('results.txt', 'submission.csv', 'artifacts.npz'):
            p = os.path.join(WORK_DIR, fn)
            if os.path.exists(p):
                shutil.copy(p, os.path.join(DRIVE_DIR, fn))
    except Exception:
        pass

# import fold caches from previous sessions (previous version outputs as input)
_cache_dst = os.path.join(CACHE_ROOT, 'foldcache_v17')
os.makedirs(_cache_dst, exist_ok=True)
for _root in ('/kaggle/input', '/content'):
    if not os.path.exists(_root): continue
    for _dn, _, _fns in os.walk(_root):
        if os.path.basename(_dn) == 'foldcache_v17':
            for _fn in _fns:
                _dst = os.path.join(_cache_dst, _fn)
                if not os.path.exists(_dst):
                    shutil.copy(os.path.join(_dn, _fn), _dst)

RESULTS = []
def log(msg):
    print(msg)
    RESULTS.append(str(msg))
    with open(os.path.join(WORK_DIR, 'results.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(RESULTS))
    _sync_to_drive()

log(f'GPU available: {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if FAST_RUN: log('FAST_RUN=True: smoke test, not a submission run')


## 2 · The official metric, replicated byte-for-byte

An exact copy of the organizer's tokenizer and alignment check. Every measurement in this notebook — and the final submission — goes through this logic.

In [ ]:
# ---------- 0) official metric logic (verbatim replica) ----------
VALID_SYMBOLS = set('.،؟!:؛-')
SYMBOLS = sorted(VALID_SYMBOLS)

def _tokenize_gold(text):
    leading, pairs, cur, in_word = [], [], [], False
    for ch in text:
        if ch.isspace():
            if in_word: pairs.append([''.join(cur), []]); cur=[]; in_word=False
            continue
        if ch in VALID_SYMBOLS:
            if in_word: pairs.append([''.join(cur), [ch]]); cur=[]; in_word=False
            else:
                if pairs: pairs[-1][1].append(ch)
                else: leading.append(ch)
            continue
        if not in_word: in_word=True; cur=[ch]
        else: cur.append(ch)
    if in_word: pairs.append([''.join(cur), []])
    return ''.join(leading), [(w, ''.join(g)) for (w,g) in pairs]

def _extract_labels(raw_text, generated_text, role):
    raw_words = str(raw_text).strip().split()
    if not raw_words: raise ValueError(f'[{role}] no words')
    _, pairs = _tokenize_gold(str(generated_text))
    gen_words = [w for (w,_) in pairs]
    if len(gen_words) != len(raw_words):
        raise ValueError(f'[{role}] word-count mismatch')
    for rw, gw in zip(raw_words, gen_words):
        if rw != gw: raise ValueError(f'[{role}] word mismatch')
    return [([c for c in gap if c in VALID_SYMBOLS] or ['0']) for _,gap in pairs]


## 3 · Load data and build the label space

An 11-class softmax space (`KEEP`) covers single marks and the frequent multi-mark gaps (`؟!` etc.), with a 7-column multi-hot view (`lab7`) for the multi-label component and the stacker. `macro_of_probs` scores any component with the official metric semantics.

In [ ]:
# ---------- 1) input files ----------
def find_file(filename):
    roots = ['/content', '/kaggle/input', '.']
    skips = [os.path.abspath(WORK_DIR), os.path.abspath('/content/drive')]
    matches = []
    for root in roots:
        if not os.path.exists(root): continue
        for dirname, dirnames, filenames in os.walk(root):
            ad = os.path.abspath(dirname)
            if any(ad.startswith(s) for s in skips):
                dirnames[:] = []; continue
            if filename in filenames:
                matches.append(os.path.join(dirname, filename))
    return sorted(matches, key=lambda p: (len(p), p))

train_path = (find_file('train.csv') or [None])[0]
test_path  = (find_file('test.csv') or [None])[0]
assert train_path and test_path, 'train.csv / test.csv not found among the inputs'
log(f'train: {train_path}\ntest:  {test_path}')
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

PBUH = 'ﷺ'
KEEP = ['', '،', '.', ':', '-', '؛', '!', '؟', '؟!', '!-', '-:']
LBL2ID = {l: i for i, l in enumerate(KEEP)}
MAP_RARE = {':-': '-:', '!-:': '!-', '؟-': '؟', '-؛': '؛', '؟!-': '؟!',
            '-،': '،', '-.': '.', '.،': '.', '!.': '!'}
LBL_MATRIX = np.zeros((len(KEEP), len(SYMBOLS)), dtype=np.int8)
for li, l in enumerate(KEEP):
    for si, s in enumerate(SYMBOLS):
        if s in l: LBL_MATRIX[li, si] = 1

def dedup_gap(g):
    seen = []
    for c in g:
        if c != '0' and c not in seen: seen.append(c)
    return ''.join(seen)

def norm_label(l):
    if l in LBL2ID: return l
    if l in MAP_RARE: return MAP_RARE[l]
    for c in l:
        if c in LBL2ID: return c
    return ''

docs = []
SYM2IDX = {s: i for i, s in enumerate(SYMBOLS)}
for _, row in train_df.iterrows():
    try:
        gaps = _extract_labels(row['text'], row['final_text'], 'gold')
    except ValueError:
        continue
    words = str(row['text']).strip().split()
    lab7 = np.zeros((len(words), 7), dtype=np.float32)
    for j, g in enumerate(gaps):
        for c in g:
            if c in SYM2IDX: lab7[j, SYM2IDX[c]] = 1.0
    docs.append({'words': words,
                 'labels': [norm_label(dedup_gap(g)) for g in gaps],
                 'lab7': lab7})
log(f'aligned documents: {len(docs)}/{len(train_df)}')
label_counts = Counter(l for d in docs for l in d['labels'])
test_words_all = [str(t).strip().split() for t in test_df['text']]

gold_flat = np.concatenate([[LBL2ID[l] for l in d['labels']] for d in docs]).astype(np.int64)
gold_bin = LBL_MATRIX[gold_flat]

def macro_of_probs(probs_flat, scales=None):
    s = scales if scales is not None else np.ones(len(KEEP))
    pb = LBL_MATRIX[np.argmax(probs_flat * s[None, :], axis=1)]
    f1s = []
    for si in range(len(SYMBOLS)):
        tp = int(((gold_bin[:,si]==1)&(pb[:,si]==1)).sum())
        fp = int(((gold_bin[:,si]==0)&(pb[:,si]==1)).sum())
        fn = int(((gold_bin[:,si]==1)&(pb[:,si]==0)).sum())
        p_ = tp/(tp+fp) if tp+fp else 0.0
        r_ = tp/(tp+fn) if tp+fn else 0.0
        f1s.append(2*p_*r_/(p_+r_) if p_+r_ else 0.0)
    return float(np.mean(f1s))


## 4 · Training machinery

Sliding windows (200 words, stride 100), labels on the last subword, class-weighted cross-entropy, center-window-wins inference, and the augmentation used by the last component: sentence recombination built **from the training fold only**, plus 3× oversampling of windows containing the rare marks. `train_family` is the workhorse — one call trains one ensemble component with n-fold CV and per-fold caching.

In [ ]:
# ---------- 2) training machinery ----------
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)

MAX_WORDS = 200
STRIDE    = 100

def make_windows(n):
    if n <= MAX_WORDS: return [(0, n)]
    wins, s = [], 0
    while True:
        e = min(s + MAX_WORDS, n)
        wins.append((s, e))
        if e == n: break
        s += STRIDE
    return wins

tokenizer = None

class PunctDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        words, labels = self.items[idx]
        enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=512)
        word_ids = enc.word_ids()
        last_sub = {}
        for pos, wid in enumerate(word_ids):
            if wid is not None: last_sub[wid] = pos
        lab = [-100] * len(word_ids)
        for wid, pos in last_sub.items():
            if labels[wid] is not None:
                lab[pos] = LBL2ID[labels[wid]]
        enc['labels'] = lab
        return {k: torch.tensor(v) for k, v in enc.items()}

def docs_to_items(dd):
    items = []
    for d in dd:
        for s, e in make_windows(len(d['words'])):
            items.append((d['words'][s:e], d['labels'][s:e]))
    return items

TERMINAL_LBL = set('.!؟')

def make_synth_docs(real_docs, factor, seed):
    """Split training docs into sentences at terminal marks, recombine randomly."""
    rng = random.Random(seed)
    sentences = []
    for d in real_docs:
        cur_w, cur_l = [], []
        for w_, l_ in zip(d['words'], d['labels']):
            cur_w.append(w_); cur_l.append(l_)
            if l_ and any(c in TERMINAL_LBL for c in l_):
                sentences.append((cur_w, cur_l)); cur_w, cur_l = [], []
        if cur_w:
            sentences.append((cur_w, cur_l))
    if not sentences:
        return []
    out = []
    for _ in range(int(factor * len(real_docs))):
        k = rng.randint(3, 12)
        ws, ls = [], []
        for _s in range(k):
            sw, sl = sentences[rng.randrange(len(sentences))]
            ws.extend(sw); ls.extend(sl)
        out.append({'words': ws, 'labels': ls})
    return out

w = np.array([1.0 / np.sqrt(max(label_counts.get(l, 1), 1)) for l in KEEP])
w = w / w.mean(); w[0] = min(w[0], 0.5)
class_weights = torch.tensor(w, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device), ignore_index=-100)
        loss = loss_fct(outputs.logits.view(-1, len(KEEP)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    labels = p.label_ids
    mask = labels != -100
    yt = LBL_MATRIX[labels[mask]]; yp = LBL_MATRIX[preds[mask]]
    f1s = []
    for si in range(len(SYMBOLS)):
        tp = int(((yt[:,si]==1)&(yp[:,si]==1)).sum())
        fp = int(((yt[:,si]==0)&(yp[:,si]==1)).sum())
        fn = int(((yt[:,si]==1)&(yp[:,si]==0)).sum())
        p_ = tp/(tp+fp) if tp+fp else 0.0
        r_ = tp/(tp+fn) if tp+fn else 0.0
        f1s.append(2*p_*r_/(p_+r_) if p_+r_ else 0.0)
    return {'macro_f1_7': float(np.mean(f1s))}

@torch.no_grad()
def predict_doc_probs(model, words, win_batch=8):
    """Window-batched inference; the most central window wins per word (~8x faster)."""
    n = len(words)
    best = [None] * n
    wins = make_windows(n)
    for i in range(0, len(wins), win_batch):
        chunk_wins = wins[i:i + win_batch]
        enc = tokenizer([words[s:e] for s, e in chunk_wins],
                        is_split_into_words=True, truncation=True,
                        max_length=512, padding=True, return_tensors='pt')
        word_ids_list = [enc.word_ids(batch_index=bi) for bi in range(len(chunk_wins))]
        enc = {k: v.to(model.device) for k, v in enc.items()}
        probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
        for bi, (s, e) in enumerate(chunk_wins):
            last_sub = {}
            for pos, wid in enumerate(word_ids_list[bi]):
                if wid is not None: last_sub[wid] = pos
            L = e - s
            for wid, pos in last_sub.items():
                gi = s + wid
                central = min(wid, L - 1 - wid)
                if best[gi] is None or central > best[gi][0]:
                    best[gi] = (central, probs[bi, pos])
    out = np.zeros((n, len(KEEP)), dtype=np.float32)
    for i, b in enumerate(best):
        if b is not None: out[i] = b[1]
        else: out[i, 0] = 1.0
    return out

def train_family(model_name, n_folds, seed_offset, tag, pseudo,
                 bs=16, lr=3e-5, ga=1, epochs=None):
    global tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    rng = random.Random(BASE_SEED)
    order = list(range(len(docs))); rng.shuffle(order)
    folds = [order[i::n_folds] for i in range(n_folds)]
    oof = [None] * len(docs)
    tps = [np.zeros((len(ws), len(KEEP)), dtype=np.float32) for ws in test_words_all]
    epochs = epochs or NUM_EPOCHS
    cache_dir = os.path.join(CACHE_ROOT, 'foldcache_v17')
    os.makedirs(cache_dir, exist_ok=True)
    def _cache_path(f): return os.path.join(cache_dir, f'{tag}_f{f}.npz')
    for fold in range(n_folds):
        cp = _cache_path(fold)
        if os.path.exists(cp):
            try:
                C = np.load(cp)
                idxs = C['idx']; oflat = C['oof']; olens = C['olens']
                assert int(max(idxs)) < len(docs) and len(C['tlens']) == len(tps)
                s = 0
                for di, L in zip(idxs, olens):
                    oof[int(di)] = oflat[s:s+L].astype(np.float32); s += L
                tflat = C['test']; tlens = C['tlens']; s = 0
                for ti, L in enumerate(tlens):
                    tps[ti] += tflat[s:s+L].astype(np.float32); s += L
                log(f'--- {tag} fold {fold+1}/{n_folds}: resumed from cache ---')
                continue
            except Exception as _ce:
                log(f'incompatible cache for {tag} fold {fold+1} — retraining ({_ce})')
        log(f'--- {tag} fold {fold+1}/{n_folds} ---')
        val_idx = set(folds[fold])
        tr_real = [docs[i] for i in range(len(docs)) if i not in val_idx]
        tr = tr_real + pseudo
        if AUGMENT_STEP:
            # synthetic docs from training-fold sentences only (no validation leakage)
            tr = tr + make_synth_docs(tr_real, AUG_SYNTH_FACTOR, BASE_SEED + seed_offset + fold)
        va = [docs[i] for i in folds[fold]]
        torch.manual_seed(BASE_SEED+seed_offset+fold); np.random.seed(BASE_SEED+seed_offset+fold)
        model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(KEEP))
        ta_kwargs = dict(
            output_dir=CKPT_DIR, num_train_epochs=epochs,
            per_device_train_batch_size=bs, per_device_eval_batch_size=32,
            gradient_accumulation_steps=ga,
            learning_rate=lr, warmup_ratio=0.1, weight_decay=0.01,
            save_strategy='epoch', save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model='macro_f1_7',
            fp16=torch.cuda.is_available(), logging_steps=200, report_to='none',
            seed=BASE_SEED+seed_offset+fold)
        if 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters:
            ta_kwargs['eval_strategy'] = 'epoch'
        else:
            ta_kwargs['evaluation_strategy'] = 'epoch'
        args = TrainingArguments(**ta_kwargs)
        tr_items = docs_to_items(tr)
        if AUGMENT_STEP:
            rare_items = [it for it in tr_items
                          if any(l is not None and ('!' in l or '؛' in l) for l in it[1])]
            tr_items = tr_items + rare_items * (AUG_RARE_DUP - 1)
        log(f'  training windows: {len(tr_items)}')
        trainer = WeightedTrainer(model=model, args=args,
                                  train_dataset=PunctDataset(tr_items),
                                  eval_dataset=PunctDataset(docs_to_items(va)),
                                  data_collator=DataCollatorForTokenClassification(tokenizer),
                                  compute_metrics=compute_metrics)
        trainer.train()
        ev = trainer.evaluate()
        log(f'{tag} fold {fold+1}: macro_f1_7={ev["eval_macro_f1_7"]:.4f}')
        model.eval()
        fold_test = []
        for di in folds[fold]:
            oof[di] = predict_doc_probs(model, docs[di]['words'])
        for ti, ws in enumerate(test_words_all):
            fp = predict_doc_probs(model, ws)
            tps[ti] += fp
            fold_test.append(fp)
        idxs = np.array(folds[fold], dtype=np.int32)
        oflat = np.concatenate([oof[int(di)] for di in idxs], axis=0).astype(np.float16)
        olens = np.array([oof[int(di)].shape[0] for di in idxs], dtype=np.int32)
        tflat = np.concatenate(fold_test, axis=0).astype(np.float16)
        tlens = np.array([f.shape[0] for f in fold_test], dtype=np.int32)
        np.savez_compressed(cp, idx=idxs, oof=oflat, olens=olens, test=tflat, tlens=tlens)
        log(f'fold {fold+1} cached ({cache_dir})')
        del trainer, model
        gc.collect(); torch.cuda.empty_cache()
        shutil.rmtree(CKPT_DIR, ignore_errors=True)
    for ti in range(len(tps)):
        tps[ti] /= n_folds
    return oof, tps


## 5 · The generation engine

The pipeline is declared as a step list: train a family → `'P'` = regenerate pseudo-labels from the mean of **all** components trained so far (confidence ≥ 0.90) → train the next family on gold + pseudo. This is exactly how the full system was grown, compressed to its four highest-value components.

In [ ]:
# ---------- 3) generation engine (from scratch — no external reservoir) ----------
reservoir_oof, reservoir_test = {}, {}
pseudo_docs = []

def regen_pseudo():
    """Pseudo-labels from the mean of all current components (confidence >= 0.90)."""
    global pseudo_docs
    if not reservoir_test:
        pseudo_docs = []; return
    pseudo_docs = []
    kept = tot = 0
    keys = list(reservoir_test)
    for ti, ws in enumerate(test_words_all):
        if not ws: continue
        probs = sum(reservoir_test[k][ti] for k in keys) / len(keys)
        ids = np.argmax(probs, axis=1)
        conf = probs[np.arange(len(ids)), ids]
        labels = [KEEP[i] if c >= CONF else None for i, c in zip(ids, conf)]
        kept += sum(l is not None for l in labels); tot += len(labels)
        pseudo_docs.append({'words': ws, 'labels': labels})
    log(f'pseudo-labels ({len(keys)} components): {kept}/{tot} ({kept/max(tot,1):.1%})')

# steps in historical order — 'P' = regenerate pseudo-labels
# softmax: (key, model, seed, nf, kw, augment)
STEPS = [
    ('S', 'ar2', M_AR, 100, 4, dict(bs=16, lr=3e-5, epochs=6), False),
    'P',
    ('S', 'ca1', M_CA, 300, 4, dict(bs=16, lr=3e-5, epochs=6), False),
    'P',
    ('S', 'ag1', M_AR, 3000, 4, dict(bs=16, lr=3e-5, epochs=8), True),
    ('M', 'ML1', M_AR, 2000, 4, dict(bs=16), False),
]

ml_outputs = {}   # ML1/ML2 -> (oof_docs, test_docs), 7 columns each
AUGMENT_STEP = False   # set per step


## 6 · The multi-label component

Instead of an 11-way softmax, this head predicts 7 independent sigmoids per word (BCE with capped pos_weight, triangular-weighted window averaging) — architecture diversity that consistently paid off in the stack.

In [ ]:
# ---------- 6) multi-label family (7 sigmoids per word) ----------
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader

class MLDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        words, lab7 = self.items[idx]   # lab7: (n,7); None rows are masked
        enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=512)
        T = len(enc['input_ids'])
        lab = np.zeros((T, 7), dtype=np.float32)
        msk = np.zeros(T, dtype=np.float32)
        prev = None
        for t, wid in enumerate(enc.word_ids()):
            if wid is not None and wid != prev and lab7[wid] is not None:
                row = lab7[wid]
                if not (isinstance(row, float) and np.isnan(row)):
                    lab[t] = row; msk[t] = 1.0
            prev = wid
        return dict(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
                    labels=lab, label_mask=msk)

def ml_collate(items):
    T = max(len(it['input_ids']) for it in items); B = len(items)
    pad = tokenizer.pad_token_id
    ii = np.full((B, T), pad, dtype=np.int64); am = np.zeros((B, T), dtype=np.int64)
    lb = np.zeros((B, T, 7), dtype=np.float32); lm = np.zeros((B, T), dtype=np.float32)
    for b, it in enumerate(items):
        L = len(it['input_ids'])
        ii[b, :L] = it['input_ids']; am[b, :L] = it['attention_mask']
        lb[b, :L] = it['labels'];    lm[b, :L] = it['label_mask']
    return (torch.from_numpy(ii), torch.from_numpy(am),
            torch.from_numpy(lb), torch.from_numpy(lm))

def ml_items(dd, use_lab7=True):
    items = []
    for d in dd:
        n = len(d['words'])
        lab7 = d['lab7'] if use_lab7 else d['_p7']
        for s, e in make_windows(n):
            items.append((d['words'][s:e], [lab7[j] for j in range(s, e)]))
    return items

@torch.no_grad()
def ml_predict(model, words, win_batch=8):
    n = len(words)
    probs = np.zeros((n, 7)); wts = np.zeros(n)
    wins = make_windows(n)
    dev = next(model.parameters()).device
    for i in range(0, len(wins), win_batch):
        chunk = wins[i:i+win_batch]
        enc = tokenizer([words[s:e] for s, e in chunk], is_split_into_words=True,
                        truncation=True, max_length=512, padding=True, return_tensors='pt')
        wid_l = [enc.word_ids(batch_index=bi) for bi in range(len(chunk))]
        enc_t = {k: v.to(dev) for k, v in enc.items() if k in ('input_ids', 'attention_mask')}
        p = torch.sigmoid(model(**enc_t).logits.float()).cpu().numpy()
        for bi, (s, e) in enumerate(chunk):
            L = e - s; prev = None
            for t, wid in enumerate(wid_l[bi]):
                if wid is None or wid == prev: prev = wid; continue
                prev = wid
                w = min(wid + 1, L - wid)
                probs[s + wid] += w * p[bi, t]; wts[s + wid] += w
    wts[wts == 0] = 1
    return (probs / wts[:, None]).astype(np.float32)

def train_ml_family(model_name, n_folds, seed_offset, tag, pseudo, bs=16):
    global tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    rng = random.Random(BASE_SEED)
    order = list(range(len(docs))); rng.shuffle(order)
    folds = [order[i::n_folds] for i in range(n_folds)]
    oof = [None] * len(docs)
    tps = [np.zeros((len(ws), 7), dtype=np.float32) for ws in test_words_all]
    cache_dir = os.path.join(CACHE_ROOT, 'foldcache_v17'); os.makedirs(cache_dir, exist_ok=True)
    # convert pseudo-labels to 7-column targets (None stays masked)
    pdocs = []
    for d in pseudo:
        n = len(d['words'])
        p7 = []
        for l in d['labels']:
            if l is None: p7.append(None)
            else:
                v = np.zeros(7, dtype=np.float32)
                for c in l:
                    if c in SYM2IDX: v[SYM2IDX[c]] = 1.0
                p7.append(v)
        pdocs.append({'words': d['words'], '_p7': p7})
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    for fold in range(n_folds):
        cp = os.path.join(cache_dir, f'{tag}_f{fold}.npz')
        if os.path.exists(cp):
            try:
                C = np.load(cp)
                assert int(max(C['idx'])) < len(docs) and len(C['tlens']) == len(tps)
                s = 0
                for di, L in zip(C['idx'], C['olens']):
                    oof[int(di)] = C['oof'][s:s+L].astype(np.float32); s += L
                s = 0
                for ti, L in enumerate(C['tlens']):
                    tps[ti] += C['test'][s:s+L].astype(np.float32); s += L
                log(f'--- {tag} fold {fold+1}/{n_folds}: resumed from cache ---')
                continue
            except Exception as _ce:
                log(f'incompatible cache for {tag} fold {fold+1} — retraining ({_ce})')
        log(f'--- {tag} fold {fold+1}/{n_folds} ---')
        torch.manual_seed(BASE_SEED+seed_offset+fold); np.random.seed(BASE_SEED+seed_offset+fold)
        val_idx = set(folds[fold])
        tr_items = ml_items([docs[i] for i in range(len(docs)) if i not in val_idx]) + \
                   ml_items(pdocs, use_lab7=False)
        model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=7).to(device)
        dl = DataLoader(MLDataset(tr_items), batch_size=bs, shuffle=True,
                        collate_fn=ml_collate, num_workers=2, pin_memory=True)
        lab_all = np.concatenate([docs[i]['lab7'] for i in range(len(docs)) if i not in val_idx])
        pos = lab_all.sum(0); neg = len(lab_all) - pos
        pw = torch.tensor(np.clip(neg/np.maximum(pos, 1), 1, 30), dtype=torch.float32, device=device)
        crit = nn.BCEWithLogitsLoss(pos_weight=pw, reduction='none')
        opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        steps = len(dl) * NUM_EPOCHS_ML
        sched = get_linear_schedule_with_warmup(opt, int(0.1*steps), steps)
        scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))
        model.train()
        for ep in range(NUM_EPOCHS_ML):
            tot, nb = 0.0, 0
            for ii, am, lb, lm in dl:
                ii, am, lb, lm = ii.to(device), am.to(device), lb.to(device), lm.to(device)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
                    logits = model(input_ids=ii, attention_mask=am).logits
                    loss = (crit(logits, lb).mean(-1) * lm).sum() / lm.sum().clamp(min=1)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                tot += loss.item(); nb += 1
            log(f'  {tag} fold {fold+1} epoch {ep+1}/{NUM_EPOCHS_ML} loss {tot/nb:.4f}')
        model.eval()
        fold_test = []
        for di in folds[fold]:
            oof[di] = ml_predict(model, docs[di]['words'])
        for ti, ws in enumerate(test_words_all):
            fp = ml_predict(model, ws)
            tps[ti] += fp; fold_test.append(fp)
        idxs = np.array(folds[fold], dtype=np.int32)
        np.savez_compressed(cp, idx=idxs,
            oof=np.concatenate([oof[int(d)] for d in idxs]).astype(np.float16),
            olens=np.array([oof[int(d)].shape[0] for d in idxs], dtype=np.int32),
            test=np.concatenate(fold_test).astype(np.float16),
            tlens=np.array([f.shape[0] for f in fold_test], dtype=np.int32))
        log(f'fold {fold+1} cached')
        del model; gc.collect(); torch.cuda.empty_cache()
    for ti in range(len(tps)):
        tps[ti] /= n_folds
    return oof, tps


## 7 · Run the generations and save the reservoir

Executes the step list, then packs all OOF/test probabilities into `artifacts.npz` (+ `probs_backup.pkl` for the multi-label component) so later sessions can stack without retraining.

In [ ]:
# ---------- 7) run the generations ----------
if not torch.cuda.is_available():
    log('!! full-training script — a GPU is required (enable T4 and rerun)')
    raise SystemExit()

for step in STEPS:
    if step == 'P':
        regen_pseudo(); continue
    kind, key, mname, soff, nf, kw, aug = step
    AUGMENT_STEP = aug
    if kind == 'S':
        oof_n, test_n = train_family(resolve_model(mname), nf, soff, key.upper(),
                                     pseudo=pseudo_docs, **kw)
        reservoir_oof[key] = oof_n; reservoir_test[key] = test_n
        log(f'{key} solo OOF={macro_of_probs(np.concatenate(oof_n, axis=0)):.4f}')
    else:
        o, t = train_ml_family(resolve_model(mname), nf, soff, key, pseudo_docs, **kw)
        ml_outputs[key] = (o, t)
AUGMENT_STEP = False

# ---------- 7b) save the full reservoir ----------
def pack(doc_arrays):
    lens = np.array([a.shape[0] for a in doc_arrays], dtype=np.int32)
    return np.concatenate(doc_arrays, axis=0).astype(np.float16), lens

save_kw = {'comp_names': np.array(list(reservoir_oof))}
for key in reservoir_oof:
    p, l = pack(reservoir_oof[key]);  save_kw[f'oof_{key}'] = p;  save_kw[f'oof_{key}_len'] = l
    p, l = pack(reservoir_test[key]); save_kw[f'test_{key}'] = p; save_kw[f'test_{key}_len'] = l
np.savez_compressed(os.path.join(WORK_DIR, 'artifacts.npz'), **save_kw)
import pickle
ML_ORDER = ['،', '.', ':', '؛', '-', '!', '؟']
PERM = [SYMBOLS.index(s) for s in ML_ORDER]
for i, key in enumerate(ml_outputs):
    o, t = ml_outputs[key]
    with open(os.path.join(WORK_DIR, f'probs_backup{"" if i == 0 else i+1}.pkl'), 'wb') as f:
        pickle.dump({'oof': [a[:, PERM] for a in o], 'test': [a[:, PERM] for a in t]}, f)
log('full reservoir saved: ' + str(list(reservoir_oof)) + ' + ' + str(list(ml_outputs)))

# assemble the stacking matrices in memory
all_names = list(reservoir_oof)
comp_oof_l  = [np.concatenate(reservoir_oof[k], axis=0).astype(np.float64) @ LBL_MATRIX.astype(np.float64) for k in all_names]
comp_test_l = [[a.astype(np.float64) @ LBL_MATRIX.astype(np.float64) for a in reservoir_test[k]] for k in all_names]
for key in ml_outputs:
    o, t = ml_outputs[key]
    comp_oof_l.append(np.concatenate(o, axis=0).astype(np.float64))
    comp_test_l.append([a.astype(np.float64) for a in t])


## 8 · Per-class logistic stacking → submission

Features per word gap: 7 symbol-probabilities from every component + the same for the previous/next gap (shifted within-document only) + 4 position flags. One logistic regression per mark, document-level 5-fold CV, per-class thresholds tuned on the CV predictions, terminal-mark rule at document end, and a row-by-row alignment self-check before writing `submission.csv`.

In [ ]:
# ---------- 8) final stacking layer + submission ----------
from sklearn.linear_model import LogisticRegression

LBL7 = LBL_MATRIX.astype(np.float64)
gold_true = np.concatenate([d['lab7'] for d in docs]).astype(np.int8)
Nn = gold_true.shape[0]
doc_id = np.concatenate([np.full(len(d['words']), i) for i, d in enumerate(docs)])
meta_rows = []
for d in docs:
    n = len(d['words']); pb = [w == PBUH for w in d['words']]
    for j in range(n):
        meta_rows.append((j == n-1, j == 0, pb[j], pb[j+1] if j+1 < n else False))
meta = np.array(meta_rows, dtype=float)

base = np.concatenate(comp_oof_l, axis=1)
def shift(F, ids, k):
    out = np.zeros_like(F)
    if k > 0:
        out[k:] = F[:-k]; out[k:][ids[k:] != ids[:-k]] = 0
    else:
        k = -k; out[:-k] = F[k:]; out[:-k][ids[:-k] != ids[k:]] = 0
    return out
X = np.concatenate([base, shift(base, doc_id, 1), shift(base, doc_id, -1), meta], axis=1)
log(f'stacking feature matrix: {X.shape}')

perm = np.random.default_rng(BASE_SEED).permutation(len(docs))
folds5 = [perm[i::5] for i in range(5)]
cvp = np.zeros((Nn, 7))
for fold in folds5:
    va = np.isin(doc_id, fold); tr = ~va
    for si in range(7):
        clf = LogisticRegression(max_iter=2000, C=1.0)
        clf.fit(X[tr], gold_true[tr, si])
        cvp[va, si] = clf.predict_proba(X[va])[:, 1]
ths = np.zeros(7); f1s = np.zeros(7)
for si in range(7):
    yb = gold_true[:, si].astype(bool); best = (0.0, 0.5)
    for t in np.arange(0.02, 0.98, 0.005):
        yp = cvp[:, si] > t
        tp = int((yb & yp).sum()); fp = int((~yb & yp).sum()); fn = int((yb & ~yp).sum())
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        if f1 > best[0]: best = (f1, t)
    f1s[si], ths[si] = best
log(f'>>> final stacking OOF (document-level CV): {f1s.mean():.4f}')
for s, f, t in zip(SYMBOLS, f1s, ths):
    log(f"  '{s}': F1={f:.4f} th={t:.3f}")

clfs = []
for si in range(7):
    clf = LogisticRegression(max_iter=2000, C=1.0)
    clf.fit(X, gold_true[:, si]); clfs.append(clf)

ORDER = ['؟', '!', '.', ':', '؛', '،', '-']
IDX_TERM = [SYMBOLS.index(c) for c in '.!؟']; IDX_DOT = SYMBOLS.index('.')
finals = []
for ti, ws in enumerate(test_words_all):
    if not ws:
        finals.append(''); continue
    n = len(ws)
    b = np.concatenate([ct[ti] for ct in comp_test_l], axis=1)
    ids = np.zeros(n, dtype=int)
    mt = np.zeros((n, 4)); pb = np.array([w == PBUH for w in ws])
    mt[:, 0] = (np.arange(n) == n-1); mt[:, 1] = (np.arange(n) == 0)
    mt[:, 2] = pb; mt[:-1, 3] = pb[1:]
    Xt = np.concatenate([b, shift(b, ids, 1), shift(b, ids, -1), mt], axis=1)
    probs = np.stack([clfs[si].predict_proba(Xt)[:, 1] for si in range(7)], axis=1)
    pbin = (probs > ths[None, :]).astype(int)
    if pbin[-1, IDX_TERM].sum() == 0: pbin[-1, IDX_DOT] = 1
    finals.append(' '.join(w + ''.join(s for s in ORDER if pbin[wi, SYMBOLS.index(s)])
                           for wi, w in enumerate(ws)))

bad = []
for i, (raw, fin) in enumerate(zip(test_df['text'], finals)):
    try:
        _extract_labels(raw, fin, 'prediction')
    except ValueError as e:
        bad.append((i, str(e)))
if bad:
    log('!! self-check FAILED: ' + str(bad[:5]))
    raise SystemExit('submission blocked by self-check')
log(f'self-check passed: all {len(finals)} rows align')
pd.DataFrame({'id': test_df['id'], 'final_text': finals}).to_csv(
    os.path.join(WORK_DIR, 'submission.csv'), index=False)
log('saved submission.csv')


log('=== done: submission.csv reproduces the final system ===')
_sync_to_drive()


---
**Outputs:** `submission.csv`, `artifacts.npz` (the 4-component reservoir), `probs_backup.pkl`, `results.txt` (full log).